In [ ]:
import os
import certifi
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
import requests 

In [ ]:
from langchain.agents import create_react_agent, AgentExecutor

In [47]:
#LOAD ENV VARIABLE
os.environ["SSL_CEERT_FILE"] = certifi.where()
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY= os.getenv("WEATHERSTACK_API_KEY")

In [ ]:
search_tool= TavilySearchResults(max_results=5)

In [57]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )
    print(url)

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}\n"
        f"Weather Descriptions: {data['current']['weather_descriptions'][0]}%"
    )

In [58]:
#Initialise LLM
llm = ChatOpenAI(
    model="gpt-4.1-nano-2025-04-14",
    temperature=0.0,
    api_key=OPENAI_API_KEY
)

In [59]:
##Langchain Predefined Prompt

from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""
)

In [60]:
##Langchain Predefined Prompt
# prompt= hub.pull("hwchase17/react",dangerously_pull_public_prompt=True)
print(prompt)

input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


In [61]:
tools= [search_tool,get_weather_data]

In [62]:
##Create Agent

agent= create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [63]:
##Define Executor
agent_executor= AgentExecutor(agent=agent, tools=tools, verbose=True,handle_parsing_errors=True)

In [68]:
##Run the agent with a query
response= agent_executor.invoke({
   "input": ("Find the Capital of Kashmir"
    "and it's current weather"
    "Predict possiblity of rain would start in next 1 hour"
   )
})



> Entering new AgentExecutor chain...
Question: Find the Capital of Kashmir and its current weather. Predict the possibility of rain that would start in the next 1 hour.  
Thought: I need to identify the capital of Kashmir first, then get its current weather, and finally assess the likelihood of rain starting within the next hour.  
Action: tavily_search_results_json  
Action Input: Capital of Kashmir  [{'title': 'Srinagar - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/Srinagar', 'content': 'Although several other capitals of Kashmir were constructed by other rulers over the next few centuries, Pravarasena\'s Srinagar survived as the primary capital. The city was divided into several parts, each with its own guardian deity, which continue to be worshipped by Hindu Kashmiris. Durlabhavardhana (625–662), founder of the Karkota dynasty, built a temple in Srinagar to celebrate his imperial campaigns in the regions neighbouring Kashmir. The 8th century scholar Adi Shankara visited th

In [69]:
print(response["output"])

The capital of Kashmir is Srinagar. Currently, it is experiencing light rain showers, and there is a high likelihood of rain starting within the next hour.
